In [39]:
import enum
from collections.abc import Callable, Iterable, Mapping, Sequence
from enum import Enum
from fractions import Fraction
from functools import partial
from typing import Any, NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse
import tqdm.notebook

from gacha_model import COND_PROB_5_STAR, COND_PROB_6_STAR, PityModel
from plot_tools import draw_multi_cdf_fig, draw_multi_pmf_cdf_fig, draw_pmf_cdf_fig
from random_variable import FiniteDist

In [47]:
# 在联动寻访中获得 UP 6★ 干员和全部 2 名 UP 5★ 干员各至少若干次所需抽数的分布

type 状态类 = 吸收态类 | 过渡态类


class 吸收态类(NamedTuple):
    """已获取 UP 6★ 干员"""
    已获取UP五星干员乙数量: int
    已获取UP五星干员丙数量: int


class 过渡态类(NamedTuple):
    """未获取 UP 6★ 干员"""
    六星水位: int
    五星水位: int
    已获取UP六星干员数量: int
    已获取UP五星干员乙数量: int
    已获取UP五星干员丙数量: int


def 获取状态(*, 六星水位: int, 五星水位: int, 已获取UP六星干员数量: int, 已获取UP五星干员乙数量: int, 已获取UP五星干员丙数量: int) -> 状态类:
    if 已获取UP六星干员数量 >= 已获取UP六星干员数量上限:
        return 吸收态类(
            已获取UP五星干员乙数量=min(已获取UP五星干员乙数量, 已获取UP五星干员乙数量上限),
            已获取UP五星干员丙数量=min(已获取UP五星干员丙数量, 已获取UP五星干员丙数量上限),
        )
    else:
        return 过渡态类(
            六星水位=六星水位,
            五星水位=五星水位,
            已获取UP六星干员数量=已获取UP六星干员数量,
            已获取UP五星干员乙数量=min(已获取UP五星干员乙数量, 已获取UP五星干员乙数量上限),
            已获取UP五星干员丙数量=min(已获取UP五星干员丙数量, 已获取UP五星干员丙数量上限),
        )


def 状态转移(起始状态: 状态类, *, 是第10抽: bool, 是第120抽: bool) -> list[tuple[状态类, float]]:
    assert not (是第10抽 and 是第120抽), "不能同时是第10抽和第120抽"

    转移概率列表: list[tuple[状态类, float]] = []

    if isinstance(起始状态, 吸收态类):
        转移概率列表.append((起始状态, 1))

    else:
        起始六星水位 = 起始状态.六星水位
        起始五星水位 = 起始状态.五星水位
        起始已获取UP六星干员数量 = 起始状态.已获取UP六星干员数量
        起始已获取UP五星干员乙数量 = 起始状态.已获取UP五星干员乙数量
        起始已获取UP五星干员丙数量 = 起始状态.已获取UP五星干员丙数量

        if 是第120抽 and 起始已获取UP六星干员数量 == 0:  # 第120抽时，若未获取过UP六星干员，则必定获取
            转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 已获取UP六星干员数量=1, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 1))

        else:
            # 计算抽到不同星级干员的概率
            六星概率 = COND_PROB_6_STAR[起始六星水位]
            if 是第10抽 and 起始五星水位 == 9:
                五星概率 = 1 - 六星概率
            else:
                五星概率 = np.clip(COND_PROB_5_STAR[起始五星水位], 0, 1 - 六星概率)
            四星或三星概率 = 1 - 六星概率 - 五星概率

            if 起始已获取UP五星干员乙数量 == 0 and 起始已获取UP五星干员丙数量 > 0:
                乙概率 = 五星概率
                丙概率 = 0
                其他五星概率 = 0
            elif 起始已获取UP五星干员乙数量 > 0 and 起始已获取UP五星干员丙数量 == 0:
                乙概率 = 0
                丙概率 = 五星概率
                其他五星概率 = 0
            else:
                乙概率 = 五星概率 / 4
                丙概率 = 五星概率 / 4
                其他五星概率 = 五星概率 / 2

            # 抽到UP6星干员
            转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量 + 1, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 六星概率 / 2))

            # 抽到其他6星干员
            转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 六星概率 / 2))

            # 抽到UP五星干员乙
            转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量 + 1, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 乙概率))

            # 抽到UP五星干员丙
            转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量 + 1), 丙概率))

            # 抽到其他五星干员
            转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 其他五星概率))

            # 抽到四星及以下干员
            转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=起始五星水位 + 1, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取UP五星干员乙数量=起始已获取UP五星干员乙数量, 已获取UP五星干员丙数量=起始已获取UP五星干员丙数量), 四星或三星概率))

    转移概率列表 = [(目标状态, 概率) for 目标状态, 概率 in 转移概率列表 if 概率 > 0]
    assert np.isclose(sum(x[1] for x in 转移概率列表), 1)
    return 转移概率列表


def 构造状态转移矩阵(状态转移: Callable[[状态类], Iterable[tuple[状态类, float]]]) -> scipy.sparse.csr_array:
    状态数量 = len(状态列表)
    row = []
    col = []
    data = []
    for 起始状态序号, 起始状态 in tqdm.notebook.tqdm(list(enumerate(状态列表))):
        转移概率列表 = 状态转移(起始状态)
        for 目标状态, 概率 in 转移概率列表:
            目标状态序号 = 状态索引[目标状态]
            row.append(起始状态序号)
            col.append(目标状态序号)
            data.append(概率)
    return scipy.sparse.csr_array((data, (row, col)), shape=(状态数量, 状态数量))


已获取UP六星干员数量上限 = 1
已获取UP五星干员乙数量上限 = 6
已获取UP五星干员丙数量上限 = 6


状态列表: list[状态类] = []
状态列表.extend(吸收态类(已获取UP五星干员乙数量=已获取UP五星干员乙数量, 已获取UP五星干员丙数量=已获取UP五星干员丙数量)
            for 已获取UP五星干员乙数量 in range(0, 已获取UP五星干员乙数量上限 + 1, 1)
            for 已获取UP五星干员丙数量 in range(0, 已获取UP五星干员丙数量上限 + 1, 1))
状态列表.extend(过渡态类(六星水位=六星水位, 五星水位=五星水位, 已获取UP六星干员数量=已获取UP六星干员数量, 已获取UP五星干员乙数量=已获取UP五星干员乙数量, 已获取UP五星干员丙数量=已获取UP五星干员丙数量)
            for 六星水位 in range(0, 99, 1)
            for 五星水位 in range(0, 40, 1)
            for 已获取UP六星干员数量 in range(0, 已获取UP六星干员数量上限, 1)
            for 已获取UP五星干员乙数量 in range(0, 已获取UP五星干员乙数量上限 + 1, 1)
            for 已获取UP五星干员丙数量 in range(0, 已获取UP五星干员丙数量上限 + 1, 1))
状态数量: int = len(状态列表)
状态索引: dict[状态类, int] = {状态: i for i, 状态 in enumerate(状态列表)}

状态转移矩阵_非第10抽且非第120抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=False, 是第120抽=False))
状态转移矩阵_第10抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=True, 是第120抽=False))
状态转移矩阵_第120抽 = 构造状态转移矩阵(partial(状态转移, 是第10抽=False, 是第120抽=True))

迭代次数 = 4096

初始状态 = 过渡态类(六星水位=0, 五星水位=0, 已获取UP六星干员数量=0, 已获取UP五星干员乙数量=0, 已获取UP五星干员丙数量=0)
当前状态分布 = np.zeros(状态数量)
当前状态分布[状态索引[初始状态]] = 1

for i in tqdm.notebook.tqdm(range(迭代次数)):
    if i == 9:
        状态转移矩阵 = 状态转移矩阵_第10抽
    elif i == 119:
        状态转移矩阵 = 状态转移矩阵_第120抽
    else:
        状态转移矩阵 = 状态转移矩阵_非第10抽且非第120抽
    当前状态分布 = 当前状态分布 @ 状态转移矩阵

吸收态序号列表 = [状态序号 for 状态序号, 状态 in enumerate(状态列表) if isinstance(状态, 吸收态类)]
过渡态序号列表 = [状态序号 for 状态序号, 状态 in enumerate(状态列表) if isinstance(状态, 过渡态类)]

吸收态概率 = np.sum(当前状态分布[吸收态序号列表])
过渡态概率 = np.sum(当前状态分布[过渡态序号列表])

print(f"吸收态概率: {吸收态概率}")
print(f"过渡态概率: {过渡态概率}")

assert np.isclose(吸收态概率, 1)
assert np.isclose(过渡态概率, 0)

已获取UP五星干员乙数量数组 = np.fromiter((状态.已获取UP五星干员乙数量 for 状态 in 状态列表), dtype=int)
已获取UP五星干员丙数量数组 = np.fromiter((状态.已获取UP五星干员丙数量 for 状态 in 状态列表), dtype=int)

获取UP6星干员时获取的乙和丙数量的联合分布 = np.zeros((已获取UP五星干员乙数量上限 + 1, 已获取UP五星干员丙数量上限 + 1))
np.add.at(获取UP6星干员时获取的乙和丙数量的联合分布, (已获取UP五星干员乙数量数组, 已获取UP五星干员丙数量数组), 当前状态分布)

assert np.isclose(np.sum(获取UP6星干员时获取的乙和丙数量的联合分布), 1)

print(获取UP6星干员时获取的乙和丙数量的联合分布)
print(1 - np.sum(获取UP6星干员时获取的乙和丙数量的联合分布[1:, 1:]))

  0%|          | 0/194089 [00:00<?, ?it/s]

  0%|          | 0/194089 [00:00<?, ?it/s]

  0%|          | 0/194089 [00:00<?, ?it/s]

  0%|          | 0/4096 [00:00<?, ?it/s]

吸收态概率: 0.9999999999999881
过渡态概率: 0.0
[[0.1609024  0.04466495 0.         0.         0.         0.
  0.        ]
 [0.04466495 0.17402999 0.08181531 0.03401588 0.0128672  0.00453718
  0.00205286]
 [0.         0.08181531 0.06803176 0.03860161 0.01814873 0.00737093
  0.00365988]
 [0.         0.03401588 0.03860161 0.02722309 0.01474187 0.00648481
  0.00337383]
 [0.         0.0128672  0.01814873 0.01474187 0.00864642 0.00396068
  0.00210536]
 [0.         0.00453718 0.00737093 0.00648481 0.00396068 0.00184898
  0.00099138]
 [0.         0.00205286 0.00365988 0.00337383 0.00210536 0.00099138
  0.00053244]]
0.2502323054097846


In [46]:
import pandas as pd
pd.DataFrame(获取UP6星干员时获取的乙和丙数量的联合分布).to_clipboard(index=False, header=False)